# Project Overview

A/B test significance was previously checked manually using online calculators, making it harder to compare multiple metrics consistently and increasing the risk of manual errors. This project replaces separate calculations with a configurable loop-based workflow that applies the same logic to every metric and produces reproducible results.

The goal is to determine whether observed differences between test groups provide enough statistical evidence to support a decision to ship, investigate further, or collect more data.

The analysis covers four conversion metrics:

add_payment_info / session
add_shipping_info / session
begin_checkout / session
new_accounts / session

Pipeline

SQL → Grain / joins check → Data validation → Observed allocation / SRM check
→ Numerator + denominator → Conversion metrics → Statistical test (proportions_ztest, α=0.05)
→ p-value + significance → Results validation → Results CSV → Tableau
→ Business interpretation → Decision (ship / investigate further / collect more data)

In [ ]:
import pandas as pd
import numpy as np

from statsmodels.stats.proportion import proportions_ztest
from google.colab import auth
from google.cloud import bigquery

In [ ]:
#Extracting A/B test data from BigQuery.
auth.authenticate_user()

PROJECT_ID = "data-analytics-mate"

client = bigquery.Client(project=PROJECT_ID)

query = """

WITH session_info AS (
    SELECT
        s.date,
        s.ga_session_id,
        sp.country,
        sp.device,
        sp.continent,
        sp.channel,
        ab.test,
        ab.test_group
    FROM `data-analytics-mate.DA.ab_test` ab
    JOIN `data-analytics-mate.DA.session` s
        ON ab.ga_session_id = s.ga_session_id
    JOIN `data-analytics-mate.DA.session_params` sp
        ON s.ga_session_id = sp.ga_session_id
),

session_with_orders AS (
    SELECT
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        COUNT(DISTINCT o.ga_session_id) AS session_with_orders
    FROM `data-analytics-mate.DA.order` o
    JOIN session_info
        ON o.ga_session_id = session_info.ga_session_id
    GROUP BY
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
),

events AS (
    SELECT
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        ep.event_name,
        COUNT(DISTINCT ep.ga_session_id) AS event_cnt
    FROM `data-analytics-mate.DA.event_params` ep
    JOIN session_info
        ON ep.ga_session_id = session_info.ga_session_id
    GROUP BY
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        ep.event_name
),

session AS (
    SELECT
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        COUNT(DISTINCT session_info.ga_session_id) AS session_cnt
    FROM session_info
    GROUP BY
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
),

account AS (
    SELECT
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group,
        COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
    FROM `data-analytics-mate.DA.account_session` acs
    JOIN session_info
        ON acs.ga_session_id = session_info.ga_session_id
    GROUP BY
        session_info.date,
        session_info.country,
        session_info.device,
        session_info.continent,
        session_info.channel,
        session_info.test,
        session_info.test_group
)

SELECT
    session_with_orders.date,
    session_with_orders.country,
    session_with_orders.device,
    session_with_orders.continent,
    session_with_orders.channel,
    session_with_orders.test,
    session_with_orders.test_group,
    'session with orders' AS event_name,
    session_with_orders.session_with_orders AS value
FROM session_with_orders

UNION ALL

SELECT
    events.date,
    events.country,
    events.device,
    events.continent,
    events.channel,
    events.test,
    events.test_group,
    events.event_name,
    events.event_cnt AS value
FROM events

UNION ALL

SELECT
    session.date,
    session.country,
    session.device,
    session.continent,
    session.channel,
    session.test,
    session.test_group,
    'session' AS event_name,
    session.session_cnt AS value
FROM session

UNION ALL

SELECT
    account.date,
    account.country,
    account.device,
    account.continent,
    account.channel,
    account.test,
    account.test_group,
    'new account' AS event_name,
    account.new_account_cnt AS value
FROM account

"""

df = client.query(query).to_dataframe(create_bqstorage_client=False)

#Loaded data preview.
df.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-06,Slovakia,mobile,Europe,Paid Search,2,2,session with orders,1
1,2020-11-06,Slovakia,mobile,Europe,Paid Search,1,2,session with orders,1
2,2020-12-09,El Salvador,mobile,Americas,Direct,4,2,session with orders,1
3,2020-12-09,El Salvador,mobile,Americas,Direct,3,2,session with orders,1
4,2020-12-21,Slovakia,mobile,Europe,Organic Search,4,2,session with orders,1


# Data Quality Check

## Grain / Joins Check

In [ ]:
# Checking grain of session_params: does each ga_session_id appear only once?
grain_check_query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT ga_session_id) AS unique_sessions
FROM `data-analytics-mate.DA.session_params`
"""

grain_check_df = client.query(grain_check_query).to_dataframe(create_bqstorage_client=False)
grain_check_df

,total_rows,unique_sessions
0,349545,349545


## Data Validation

Basic validation of key fields, duplicates, event coverage, and data types.

The checks confirm that the data required for the A/B test analysis is complete and correctly formatted before further calculations.

In [ ]:
# Checking for missing values in key columns
df[["test", "test_group", "event_name", "value"]].isna().sum()

,0
test,0
test_group,0
event_name,0
value,0


In [ ]:
# Checking for fully duplicated rows
df.duplicated().sum()

np.int64(0)

In [ ]:
# Checking that all expected event names are present in the data
expected_events = {
    "add_payment_info",
    "add_shipping_info",
    "begin_checkout",
    "session",
    "new account",
    "session with orders"
}

missing_events = expected_events - set(df["event_name"].unique())

missing_events

set()

In [ ]:
# Confirming that 'value' is numeric
df["value"].dtype

Int64Dtype()

In [ ]:
# Checking unique values in test and test_group
df["test"].unique(), df["test_group"].unique()

(<IntegerArray>
 [2, 1, 4, 3]
 Length: 4, dtype: Int64,
 <IntegerArray>
 [2, 1]
 Length: 2, dtype: Int64)

## Observed Allocation / SRM Check

Observed Control/Test allocation is reported descriptively for each experiment.
This is not a formal Sample Ratio Mismatch test, since the expected traffic allocation
ratio for each test is not documented in the available project materials.

In [ ]:
# Observed allocation between test groups (descriptive only — not a formal SRM test,
# since the expected allocation ratio for each experiment is not documented)
allocation_results = []

for test_number in df["test"].dropna().unique():

    group_sessions = {}

    for group in [1, 2]:
        sessions = df[
            (df["test"] == test_number) &
            (df["test_group"] == group) &
            (df["event_name"] == "session")
        ]["value"].sum()

        group_sessions[group] = sessions

    total = group_sessions[1] + group_sessions[2]

    allocation_results.append({
        "test_number": test_number,
        "group_1_sessions": group_sessions[1],
        "group_2_sessions": group_sessions[2],
        "group_1_share": group_sessions[1] / total,
        "group_2_share": group_sessions[2] / total,
        "observed_ratio": group_sessions[2] / group_sessions[1]
    })

pd.DataFrame(allocation_results)

,test_number,group_1_sessions,group_2_sessions,group_1_share,group_2_share,observed_ratio
0,2,50637,50244,0.501948,0.498052,0.992239
1,1,45362,45193,0.500933,0.499067,0.996274
2,4,105079,105141,0.499853,0.500147,1.000590
3,3,70047,70439,0.498605,0.501395,1.005596


# Metric Configuration


In [ ]:
# Fixed metric definitions: metric name + which events form its numerator/denominator
metrics = [
    {
        "metric": "add_payment_info / session",
        "numerator_event": "add_payment_info",
        "denominator_event": "session"
    },
    {
        "metric": "add_shipping_info / session",
        "numerator_event": "add_shipping_info",
        "denominator_event": "session"
    },
    {
        "metric": "begin_checkout / session",
        "numerator_event": "begin_checkout",
        "denominator_event": "session"
    },
    {
        "metric": "new_accounts / session",
        "numerator_event": "new account",
        "denominator_event": "session"
    }
]

# Statistical Testing


## Intermediate Results

In [ ]:
# Building intermediate results: numerator, denominator, conversion per test/metric/group
intermediate_results = []

for test_number in df["test"].dropna().unique():

    for metric in metrics:

        metric_name = metric["metric"]
        numerator_event = metric["numerator_event"]
        denominator_event = metric["denominator_event"]

        group_results = {}

        for group in [1, 2]:

            numerator = df[
                (df["test"] == test_number) &
                (df["test_group"] == group) &
                (df["event_name"] == numerator_event)
            ]["value"].sum()

            denominator = df[
                (df["test"] == test_number) &
                (df["test_group"] == group) &
                (df["event_name"] == denominator_event)
            ]["value"].sum()

            conversion = numerator / denominator if denominator > 0 else None

            group_results[group] = {
                "numerator": numerator,
                "denominator": denominator,
                "conversion": conversion
            }

        intermediate_results.append({
            "test_number": test_number,
            "metric": metric_name,

            "group_1_numerator": group_results[1]["numerator"],
            "group_1_denominator": group_results[1]["denominator"],
            "group_1_conversion": group_results[1]["conversion"],

            "group_2_numerator": group_results[2]["numerator"],
            "group_2_denominator": group_results[2]["denominator"],
            "group_2_conversion": group_results[2]["conversion"]
        })

In [ ]:
# Filtering out rows where either group has zero denominator (can't compute a rate)
valid_results = [
    result for result in intermediate_results
    if result["group_1_denominator"] > 0
    and result["group_2_denominator"] > 0
]

dropped = len(intermediate_results) - len(valid_results)
print(f"Dropped {dropped} rows due to zero denominator")

Dropped 0 rows due to zero denominator


In [ ]:
# Sanity check: numerator should not exceed denominator, conversion should be within [0, 1]
invalid_rows = [
    result for result in valid_results
    if not (0 <= result["group_1_conversion"] <= 1)
    or not (0 <= result["group_2_conversion"] <= 1)
    or result["group_1_numerator"] > result["group_1_denominator"]
    or result["group_2_numerator"] > result["group_2_denominator"]
]

print(f"Found {len(invalid_rows)} rows failing sanity checks")
invalid_rows

Found 0 rows failing sanity checks


[]

## Statistical Test

In [ ]:
# Statistical testing for each test and metric
final_results = []

for result in valid_results:

    count = [
        result["group_1_numerator"],
        result["group_2_numerator"]
    ]

    nobs = [
        result["group_1_denominator"],
        result["group_2_denominator"]
    ]

    z_stat, p_value = proportions_ztest(
        count=count,
        nobs=nobs,
        alternative="two-sided"
    )

    final_results.append({
        **result,
        "z_stat": z_stat,
        "p_value": p_value,
        "significant": p_value < 0.05
    })

# Preview of final results
pd.DataFrame(final_results)

,test_number,metric,group_1_numerator,group_1_denominator,group_1_conversion,group_2_numerator,group_2_denominator,group_2_conversion,z_stat,p_value,significant
0,2,add_payment_info / session,1096,50637,0.021644,1125,50244,0.022391,-0.807895,0.419151,False
1,2,add_shipping_info / session,2194,50637,0.043328,2136,50244,0.042513,0.638943,0.522860,False
2,2,begin_checkout / session,2195,50637,0.043348,2137,50244,0.042532,0.638682,0.523030,False
3,2,new_accounts / session,4165,50637,0.082252,4184,50244,0.083274,-0.588793,0.556000,False
4,1,add_payment_info / session,964,45362,0.021251,983,45193,0.021751,-0.518551,0.604074,False
5,1,add_shipping_info / session,1860,45362,0.041003,1992,45193,0.044078,-2.291930,0.021910,True
6,1,begin_checkout / session,1862,45362,0.041048,1992,45193,0.044078,-2.258498,0.023915,True
7,1,new_accounts / session,3823,45362,0.084278,3681,45193,0.081451,1.542883,0.122859,False
8,4,add_payment_info / session,1850,105079,0.017606,1841,105141,0.017510,0.167535,0.866949,False
9,4,add_shipping_info / session,2661,105079,0.025324,2667,105141,0.025366,-0.061455,0.950996,False


## Results Validation

In [ ]:
# Sanity checks on the final statistical output
expected_rows = df["test"].dropna().nunique() * len(metrics)
print(f"Expected rows: {expected_rows}, Actual rows: {len(final_results)}")

invalid_p = [r for r in final_results if not (0 <= r["p_value"] <= 1)]
print(f"Rows with invalid p_value: {len(invalid_p)}")

invalid_z = [r for r in final_results if not np.isfinite(r["z_stat"])]
print(f"Rows with invalid z_stat: {len(invalid_z)}")

inconsistent_flag = [r for r in final_results if r["significant"] != (r["p_value"] < 0.05)]
print(f"Rows with inconsistent significant flag: {len(inconsistent_flag)}")

pairs = [(r["test_number"], r["metric"]) for r in final_results]
duplicate_pairs = len(pairs) - len(set(pairs))
print(f"Duplicate test_number + metric pairs: {duplicate_pairs}")

Expected rows: 16, Actual rows: 16
Rows with invalid p_value: 0
Rows with invalid z_stat: 0
Rows with inconsistent significant flag: 0
Duplicate test_number + metric pairs: 0


# Export Results

In [ ]:
# Exporting final statistical results for Tableau
results_df = pd.DataFrame(final_results)

output_file = "ab_test_significance_results.csv"
results_df.to_csv(output_file, index=False)

print(f"Exported {len(results_df)} rows to {output_file}")
results_df.head()

Exported 16 rows to ab_test_significance_results.csv


,test_number,metric,group_1_numerator,group_1_denominator,group_1_conversion,group_2_numerator,group_2_denominator,group_2_conversion,z_stat,p_value,significant
0,2,add_payment_info / session,1096,50637,0.021644,1125,50244,0.022391,-0.807895,0.419151,False
1,2,add_shipping_info / session,2194,50637,0.043328,2136,50244,0.042513,0.638943,0.522860,False
2,2,begin_checkout / session,2195,50637,0.043348,2137,50244,0.042532,0.638682,0.523030,False
3,2,new_accounts / session,4165,50637,0.082252,4184,50244,0.083274,-0.588793,0.556000,False
4,1,add_payment_info / session,964,45362,0.021251,983,45193,0.021751,-0.518551,0.604074,False


In [ ]:
# Downloading the file locally (optional — only needed if not accessing via Drive)
from google.colab import files
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Business Interpretation

In [ ]:
# Adding absolute and relative change for interpretation
business_results = results_df.copy()

business_results["absolute_change"] = (
    business_results["group_2_conversion"]
    - business_results["group_1_conversion"]
)

business_results["relative_change"] = (
    business_results["absolute_change"]
    / business_results["group_1_conversion"]
)

business_results[
    [
        "test_number",
        "metric",
        "group_1_conversion",
        "group_2_conversion",
        "absolute_change",
        "relative_change",
        "p_value",
        "significant"
    ]
]

,test_number,metric,group_1_conversion,group_2_conversion,absolute_change,relative_change,p_value,significant
0,2,add_payment_info / session,0.021644,0.022391,0.000746,0.034489,0.419151,False
1,2,add_shipping_info / session,0.043328,0.042513,-0.000815,-0.018821,0.522860,False
2,2,begin_checkout / session,0.043348,0.042532,-0.000815,-0.018809,0.523030,False
3,2,new_accounts / session,0.082252,0.083274,0.001022,0.012419,0.556000,False
4,1,add_payment_info / session,0.021251,0.021751,0.000500,0.023523,0.604074,False
5,1,add_shipping_info / session,0.041003,0.044078,0.003074,0.074973,0.021910,True
6,1,begin_checkout / session,0.041048,0.044078,0.003030,0.073818,0.023915,True
7,1,new_accounts / session,0.084278,0.081451,-0.002827,-0.033543,0.122859,False
8,4,add_payment_info / session,0.017606,0.017510,-0.000096,-0.005452,0.866949,False
9,4,add_shipping_info / session,0.025324,0.025366,0.000042,0.001664,0.950996,False


# Conclusions

Test 1 shows a moderate result: 2 of 4 metrics are significant (add_shipping_info p=0.022, begin_checkout p=0.024), while add_payment_info (p=0.604) and new_accounts (p=0.123) are not. The signal is limited to the shipping/checkout steps — worth a closer look there specifically before any rollout decision.

Test 2 shows no significant differences on any metric (p-values from 0.42 to 0.56). This isn't evidence the change failed — just that this data doesn't confirm an effect either way. Best treated as inconclusive.

Test 3 shows no significant differences on any metric (p-values from 0.52 to 0.90). No evidence of an effect here either.

Test 4 has one significant metric (new_accounts, p=0.018), with the rest far from the threshold (p=0.87–0.94). A single hit out of four comparisons is a weak signal — worth scrutiny before treating it as confirmed, especially given the multiple comparisons risk noted below.

Overall, none of the four tests shows a strong, consistent signal across multiple metrics. Test 1 has the most significant results (2 of 4), but they're limited to two related steps, not the full funnel. Tests 2 and 3 show no evidence of an effect, and Test 4 shows a single result that needs scrutiny. Significance here means evidence of a difference — not, by itself, a decision to ship.

Note on multiple comparisons: Each test checks 4 metrics at once, which increases the chance of a significant result appearing by chance alone. This is relevant for Test 1's two significant results and Test 4's single one — both warrant retesting or a longer observation window before being treated as confirmed effects

## Decision (Accept / Reject / Partially Accept / Reject and Iterate)

In [ ]:
# Decision (Accept / Reject / Partially Accept / Reject and Iterate)
decision_summary = []

for test_number in sorted(business_results["test_number"].unique()):
    test_rows = business_results[business_results["test_number"] == test_number]
    sig_count = test_rows["significant"].sum()
    total = len(test_rows)

    if sig_count == total:
        decision = "Accept"
    elif sig_count == 0:
        decision = "Reject (inconclusive — insufficient evidence, not evidence of harm)"
    elif sig_count == 1:
        decision = "Reject and Iterate"
    else:
        decision = "Partially Accept"

    decision_summary.append({
        "test_number": test_number,
        "significant_metrics": f"{sig_count}/{total}",
        "decision": decision
    })

pd.DataFrame(decision_summary)

,test_number,significant_metrics,decision
0,1,2/4,Partially Accept
1,2,0/4,"Reject (inconclusive — insufficient evidence, ..."
2,3,0/4,"Reject (inconclusive — insufficient evidence, ..."
3,4,1/4,Reject and Iterate


**Recommendation:** None of the four tests show strong, unambiguous evidence of a positive effect. Test 1 is a partial case — the shipping/checkout signal (2 of 4 metrics) supports "Partially Accept" for those specific steps, with retesting on the other two. Tests 2 and 3 are best treated as Reject — inconclusive, not evidence of harm. Test 4 falls under Reject and Iterate — one weak signal worth another look, not yet actionable.

# Tableau Dashboard

Interactive dashboard built on the statistical results from this notebook.

**What it shows:**

- 4 conversion metrics with Group 1 / Group 2 rates and absolute change (pp)
- Statistical significance for each metric (p-value < 0.05)
- Filter by test number (Test 1–4)
- Optional filters: Date, Country, Device

**Logic:**
Significance is taken directly from the Python output:
`significant = p_value < 0.05` (two-sided proportions z-test).
**Dashboard link:**  
[Tableau Public — A/B Testing Statistical Significance](https://public.tableau.com/views/ABTESTINGSTATISTICALSIGNIFICANCE/ABTESTINGSTATISTICALSIGNIFICANCE)